In [1]:
!pip install --upgrade google-cloud-aiplatform google-adk litellm requests

  Using cached litellm-1.89.3-py3-none-any.whl.metadata (34 kB)
Using cached litellm-1.89.3-py3-none-any.whl (15.5 MB)
  Attempting uninstall: litellm
    Found existing installation: litellm 1.83.7
    Uninstalling litellm-1.83.7:
      Successfully uninstalled litellm-1.83.7


In [2]:
!pip install google-adk[extensions]

  Using cached litellm-1.83.14-py3-none-any.whl.metadata (33 kB)
  Using cached openai-2.24.0-py3-none-any.whl.metadata (29 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
INFO: pip is looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
  Using cached litellm-1.83.13-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.12-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.11-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.10-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.9-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.8-py3-none-any.whl.metadata (33 kB)
  Using cached litellm-1.83.7-py3-none-any.whl.metadata (31 kB)
Using cached litellm-1.83.7-py3-none-any.whl (16.1 MB)
  Attempting uninstall: litellm
    Found existing installation: litellm 1.89.3
    Uninstalling litellm-1.89.3:
      Successfully uninstalled litellm-

In [ ]:
import os
import requests
from typing import Tuple, Dict, Any, Optional, List

# Imports from the Gemini Agent Development Kit (ADK) and LiteLLM
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY")
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "YOUR_GOOGLE_MAPS_API_KEY")

In [4]:
def get_lat_lon(address: str) -> Optional[Tuple[float, float]]:
    """
    Convert a textual address or city name into latitude and longitude
    using the Google Maps Geocoding API.

    Args:
        address (str): The string representing the location (e.g., "Los Angeles, CA").

    Returns:
        Optional[Tuple[float, float]]: A tuple containing (latitude, longitude)
        if successful. Returns None if an error occurs.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": GOOGLE_MAPS_API_KEY
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        if data["status"] == "OK":
            location = data["results"][0]["geometry"]["location"]
            return location["lat"], location["lng"]
        else:
            print(f"Geocoding error: {data['status']}")
            return None

    except requests.RequestException as e:
        print(f"API Request failed: {e}")
        return None

In [5]:
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast dictionaries.
        Returns None if data is unavailable or an error occurs.
    """
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    headers = {"User-Agent": "(myweatheragent.com, contact@example.com)"}

    try:
        response = requests.get(points_url, headers=headers)
        response.raise_for_status()
        points_data = response.json()

        forecast_url = points_data["properties"]["forecast"]

        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()

        # Light transformation to enforce the expected List[Dict[str, str]] format
        periods = forecast_data["properties"]["periods"]
        cleaned_periods = []
        for period in periods:
            cleaned_periods.append({
                "name": str(period.get("name", "")),
                "temperature": f"{period.get('temperature', '')} {period.get('temperatureUnit', '')}",
                "detailedForecast": str(period.get("detailedForecast", ""))
            })

        return cleaned_periods

    except requests.RequestException as e:
        print(f"NWS API Request failed: {e}")
        return None

In [6]:
WEATHER_AGENT_INSTRUCTIONS = """You are Pat, a friendly weather agent. Your job is to provide accurate weather forecasts for US cities.
To answer a user's request, follow these steps strictly:
1. Always use the `get_lat_lon` tool first to find the exact latitude and longitude of the city requested by the user.
2. Pass those exact coordinates into the `get_extended_weather_forecast` tool to get the current weather data.
3. Summarize the weather forecast clearly and cheerfully for the user, mentioning the temperature and general conditions.
Only use the tools provided to look up information."""

# List of tools made available to the agents
weather_tools = [get_extended_weather_forecast, get_lat_lon]

In [7]:
# Pat configured with the ADK's native Gemini model
weather_agent = Agent(
    name="Pat",
    model="gemini-2.5-flash",
    description="Pat the Friendly Weather Agent.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=weather_tools
)

In [14]:
groq_weather_agent = Agent(
    name="Pat_Groq",
    model=LiteLlm(model="groq/llama-3.1-8b-instant"),
    description="Pat the Friendly Weather Agent (powered by Llama 3.1 8B Instant via Groq).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=weather_tools,
)

In [17]:
import logging
import os
import threading
import time
import warnings

import litellm
from vertexai.preview import reasoning_engines

# 1) Silence ADK / LiteLLM internal loggers AND LiteLLM's hardcoded print()
#    spam ("Give Feedback / Get Help") that bypasses the logger.
for _name in ("google_adk", "google.adk", "LiteLLM", "litellm"):
    logging.getLogger(_name).setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
litellm.suppress_debug_info = True

# 2) ADK runs the LLM in a worker thread; Python prints unhandled thread
#    exceptions via threading.excepthook BEFORE they propagate back to the
#    main thread. Override the hook so the per-call try/except below is the
#    single source of error reporting.
threading.excepthook = lambda args: None

# 3) Disable LiteLLM auto-retry: when we hit a Groq rate limit, retrying
#    inside the same TPM window just burns more tokens and triggers more 429s.
#    We pace manually below instead.
litellm.num_retries = 0

# Groq free tier = 6000 tokens/min on llama-3.1-8b-instant.
GROQ_REQUEST_GAP_SECONDS = 8

app = reasoning_engines.AdkApp(agent=weather_agent)
app_groq = reasoning_engines.AdkApp(agent=groq_weather_agent)

test_user = "test-runner"


def _extract_session_id(session_obj):
    return session_obj.get("session_id") if isinstance(session_obj, dict) else getattr(session_obj, "id", None)


def _short_error(exc: Exception) -> str:
    """Return only the most useful one-line summary of a deep ADK exception."""
    msg = str(exc).strip().splitlines()[-1] if str(exc).strip() else type(exc).__name__
    return msg[:300]


def _run_agent_tests(
    adk_app,
    agent_label,
    session_id,
    prompts,
    delay_seconds=0,
    fresh_session_per_prompt=False,
):
    """Run each prompt against the agent.

    When fresh_session_per_prompt=True, each prompt uses a brand-new ADK session
    so conversation history does not accumulate. This keeps per-request token
    count low and avoids hitting Groq's free-tier 6000 TPM ceiling.
    """
    current_session_id = session_id
    for i, prompt in enumerate(prompts):
        if i > 0 and delay_seconds:
            time.sleep(delay_seconds)
        if fresh_session_per_prompt:
            try:
                current_session_id = _extract_session_id(
                    adk_app.create_session(user_id=test_user)
                )
            except Exception:
                current_session_id = session_id
        print(f"\n[User -> {agent_label}]: {prompt}")
        try:
            response_text = ""
            for event in adk_app.stream_query(
                user_id=test_user,
                session_id=current_session_id,
                message=prompt
            ):
                if "content" in event and "parts" in event["content"]:
                    for part in event["content"]["parts"]:
                        if "text" in part:
                            response_text += part["text"]
            if response_text.strip():
                print(f"[{agent_label}]:\n{response_text}")
            else:
                print(f"[{agent_label}]: (no text returned)")
        except Exception as e:
            print(f"[{agent_label}] FAILED: {_short_error(e)}")


try:
    session_gemini_id = _extract_session_id(app.create_session(user_id=test_user))
except Exception as e:
    print(f"Could not create Gemini session: {e}")
    session_gemini_id = "fallback-session-id"

try:
    session_groq_id = _extract_session_id(app_groq.create_session(user_id=test_user))
except Exception as e:
    print(f"Could not create Groq session: {e}")
    session_groq_id = "fallback-session-id"

test_cities = ["New York, NY", "Seattle, WA", "Miami, FL"]
test_prompts = [f"Hi Pat! What is the weather like in {c}?" for c in test_cities]

print("=" * 60)
print("=== TEST 1: Native Gemini Agent (Pat) ===")
print("=" * 60)
_run_agent_tests(app, weather_agent.name, session_gemini_id, test_prompts)

print("\n" + "=" * 60)
print("=== TEST 2: Third-Party Model via LiteLLM (Pat-Groq) ===")
print("=" * 60)
print("Note: requires a valid GROQ_API_KEY env var (free tier at https://console.groq.com).")
print(
    f"Pacing requests by {GROQ_REQUEST_GAP_SECONDS}s with a fresh session per "
    "prompt to stay under the free-tier 6000 TPM limit."
)
_run_agent_tests(
    app_groq,
    groq_weather_agent.name,
    session_groq_id,
    test_prompts,
    delay_seconds=GROQ_REQUEST_GAP_SECONDS,
    fresh_session_per_prompt=True,
)

=== TEST 1: Native Gemini Agent (Pat) ===

[User -> Pat]: Hi Pat! What is the weather like in New York, NY?
[Pat]:
Hello there! The weather in New York, NY for tonight is a low around 64 degrees Fahrenheit, with temperatures rising to around 66 overnight. There's a 60% chance of rain showers before 8 PM, and it will be mostly cloudy with new rainfall amounts less than a tenth of an inch possible.

[User -> Pat]: Hi Pat! What is the weather like in Seattle, WA?
[Pat]:
Hello there! In Seattle, WA, the weather for this afternoon is mostly cloudy with a high near 85 degrees Fahrenheit. There will be a north northwest wind around 8 mph.

[User -> Pat]: Hi Pat! What is the weather like in Miami, FL?
[Pat]:
Hello there! The weather in Miami, FL, tonight will be mostly cloudy with a low around 82 degrees Fahrenheit. There's a slight chance of showers and thunderstorms before 8 PM, followed by some patchy smoke. The heat index could reach as high as 100 degrees, so it will feel quite warm!

===